In [ ]:
path_prefix = '../..'

from collections import Counter
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Analysis

In [ ]:
dataset_path = f'{path_prefix}/data/hallusion_bench/hallusion_bench.json'

with open(dataset_path, 'r') as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)
print(f'Length of the dataset: {len(df)}')
df.head()

In [ ]:
category_counts = Counter([data['category'] for data in dataset])
gt_answer_counts = Counter([data['gt_answer'] for data in dataset])
print(f'Category counter: {dict(category_counts)}')
print(f'GT answer counter: {dict(gt_answer_counts)}')

In [ ]:
gt_answer_details_lengths = [len(data['gt_answer_details']) for data in dataset]
gt_answer_details_counts = Counter(gt_answer_details_lengths)

print(f'gt_answer_details lengths mean: {round(np.mean(gt_answer_details_lengths), 2)}')
print(f'gt_answer_details lengths median: {round(np.median(gt_answer_details_lengths), 2)}')
print(f'gt_answer_details lengths q1: {round(np.quantile(gt_answer_details_lengths, 0.25), 2)}')
print(f'gt_answer_details lengths q3: {round(np.quantile(gt_answer_details_lengths, 0.75), 2)}')

plt.figure()
plt.bar(gt_answer_details_counts.keys(), gt_answer_details_counts.values())
plt.xlabel('Length of gt_answer_details')
plt.ylabel('Frequency')
plt.title('Distribution of gt_answer_details String Lengths')
plt.show()

## Dataset Construction

In [ ]:
q1 = np.quantile([len(data['gt_answer_details']) for data in dataset], 0.25)

df_filtered = df[df['gt_answer_details'].str.len() > q1]
df_filtered = df_filtered[df_filtered['visual_input'] != '0']
print(f'Length of the filtered dataset: {len(df_filtered)}')
df_filtered.head()

In [ ]:
category_counts = Counter(df_filtered['category'])
gt_answer_counts = Counter(df_filtered['gt_answer'])
print(f'Category counter: {dict(category_counts)}')
print(f'GT answer counter: {dict(gt_answer_counts)}')

In [ ]:
stratify_cols = ['gt_answer', 'category']
df_sample = (df_filtered
             .groupby(stratify_cols, group_keys=False)
             .apply(lambda x: x.sample(n=min(125, len(x)), random_state=42), include_groups=True)
             .reset_index(drop=True))

print(f'Shape: {df_sample.shape}')
print(f'gt_answer: {df_sample['gt_answer'].value_counts(normalize=True).round(3)}')
print(f'category: {df_sample['category'].value_counts(normalize=True).round(3)}')
print(f'subcategory: {df_sample['subcategory'].value_counts(normalize=True).round(3)}')
df_sample = df_sample.reset_index().rename(columns={'index': 'id'})
df_sample.head()

In [ ]:
output = df_sample.to_dict('records')
with open(f'{path_prefix}/data/hallusion_bench/hallusion_bench_sample.json', 'w') as f:
    json.dump(output, f, indent=4)